In [1]:
import json
import os
import subprocess
import time
import math
import statistics
from typing import List, Optional
from common import *
from experiment import NetworkSetting
from data.multicast import MulticastExperiment
from treatments.multicast import treatment_map
from treatments.network_settings import *

# Collect Data

In [2]:
labels = [
    'baseline',
    f'iblt_delay{ACK_DELAY_CELL}_hint_nack',
]

NUM_CLIENTS = [1, 2, 4, 8, 16]
TREATMENTS = [treatment_map[label] for label in labels]

DURATION = 180

In [3]:
# data packet every 20 ms
treatments = [t for t in TREATMENTS if 'iblt' not in t.label() or str(ACK_DELAY_WIFI) in t.label()]
exp = MulticastExperiment(DURATION, TREATMENTS, NETWORK_SETTING_WIFI_MCAST, num_clients=NUM_CLIENTS)
data = exp.to_raw_data(execute=True)
results_wifi = data.data

In [4]:
treatments = [t for t in TREATMENTS if 'iblt' not in t.label() or str(ACK_DELAY_CELL) in t.label()]
exp = MulticastExperiment(DURATION, TREATMENTS, NETWORK_SETTING_CELLULAR_MCAST, num_clients=NUM_CLIENTS)
data = exp.to_raw_data(execute=True)
results_cell = data.data

In [5]:
treatments = [t for t in TREATMENTS if 'iblt' not in t.label() or str(ACK_DELAY_SAT) in t.label()]
exp = MulticastExperiment(DURATION, TREATMENTS, NETWORK_SETTING_SAT_MCAST, num_clients=NUM_CLIENTS)
data = exp.to_raw_data(execute=True)
results_sat = data.data

In [6]:
# check results
def check_multicast_output(output, num_clients):
    assert output.get('success')
    assert len(output.get('client_ids')) == num_clients
    assert len(output.get('latencies')) == num_clients
    assert len(output.get('num_spurious')) == num_clients

def check_outputs(results, label):
    for label in results:
        for num_clients in results[label]:
            print(label, num_clients)
            check_multicast_output(results[label][num_clients], num_clients)
    print(f"all good - {label}")

# check_outputs(results_wifi, 'wifi')
# check_outputs(results_cell, 'cellular')
# check_outputs(results_sat, 'satellite')

# Plotting Functions

In [7]:
def preprocess_latencies(latencies, num_ticks=100):
    ticks = []
    latencies.sort()
    for i in range(num_ticks):
        index = int(i * len(latencies) / num_ticks)
        ticks.append(latencies[index])
    ticks.append(latencies[-1])
    # Convert ns to ms
    return [tick / 1000000.0 for tick in ticks]

# Plot num clients vs. rx_packets

In [8]:
def network_metric(output, iface='h0-eth0', metric='rx_packets'):
    assert output['success']
    index = output['statistics']['ifaces'].index(iface)
    value = output['statistics'][metric][index] / output['time_s']
    return value

def num_nacks(output):
    assert output['success']
    print(output['num_nacks'])
    return sum(output['num_nacks'])

In [9]:
def plot_num_clients_vs_metric(data, labels, metric_func=network_metric, metric='rx_packets', normalize=True, title=None, ncol=3, ylim=0, delta=25, pdf=None):
    plt.figure(figsize=(6, 4))

    # Plot each label
    for label in labels:
        xs = []
        ys = []

        treatment_data = data[label]
        for num_clients in sorted(treatment_data.keys()):
            output = treatment_data.get(num_clients)
            if not output:
                continue
            xs.append(num_clients)
            value = metric_func(output)
            if normalize:
                value /= num_clients
            ys.append(value)

        plt.plot(xs, ys, marker='.', label=label)

    plt.title(title)
    plt.xlabel('num multicast clients')
    if normalize:
        plt.ylabel(f'{metric} / s / client')
    else:
        plt.ylabel(f'{metric} / s')
    plt.grid()
    plt.xlim(0)
    plt.ylim(ylim)
    plot_title_and_legend(title, labels, base_height=1.13, row_height=0.07, title_height=0.08, ncol=ncol)
    if pdf:
        save_pdf(pdf)
    plt.show()

def plot_num_clients_vs_metric_bar(
    data, keys, normalize=True, metric='End-to-End\nNACKs', style=False,
    title=None, ncol=3, ylim=0, delta=25, pdf=None, metric_func=num_nacks,
):
    plt.figure(figsize=(6, 1.5))

    # Get all unique num_clients values across all treatments
    all_num_clients = sorted(set(
        num_clients
        for key in keys
        for num_clients in data[key].keys()
    ))
    x_indices = np.arange(len(all_num_clients))
    bar_width = 0.8 / len(keys)

    labels = []
    for i, key in enumerate(keys):
        ys = []
        for num_clients in all_num_clients:
            output = data[key].get(num_clients)
            if not output or not output['success']:
                ys.append(0)
                continue
            value = metric_func(output)
            if normalize:
                value /= num_clients
            ys.append(value)

        offset = (i - len(keys) / 2) * bar_width + bar_width / 2
        if style:
            label = LABEL_MAP[key]
            sty = STYLE[label]
            bars = plt.bar(x_indices + offset, ys, width=bar_width, label=label, color=sty.color)
        else:
            label = key
            bars = plt.bar(x_indices + offset, ys, width=bar_width, label=label)
        labels.append(label)

        # Add value labels on top of each bar
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                plt.text(
                    bar.get_x() + bar.get_width() / 2,
                    height,
                    f'{height:.1f}',
                    ha='center', va='bottom', fontsize=10
                )

    plt.xticks(x_indices, all_num_clients)
    plt.xlabel('Number of Multicast Clients')
    if normalize:
        plt.ylabel(f'{metric} / Client / s')
    else:
        plt.ylabel(f'{metric} / s')
    plt.grid(axis='y')
    plt.ylim(ylim)
    plot_title_and_legend(title, labels, base_height=1.35, row_height=0.07, title_height=0.06, ncol=ncol)
    if pdf:
        save_pdf(pdf)
    # plt.tight_layout()
    plt.show()

In [ ]:
plot_num_clients_vs_metric_bar(results_cell, keys=labels, metric_func=num_nacks, normalize=True, style=True, ylim=(0, 1200),
                                pdf='../figures/multicast_benchmark.pdf')